# Managing Long Conversations

This project builds middleware, which allows you intercept and customize an agent's execution at every step. It's a catchall term for functions that we can use to insert capabilities in the agents execution.

The focus is on building logic to managing long-running conversations. Trimming, compression and storing information from messages to manage the context window and cost.

In [7]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware, before_agent
from langchain.messages import HumanMessage, AIMessage, RemoveMessage, ToolMessage
from typing import Any
from langgraph.runtime import Runtime

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

## Trimming and Summarizing Conversations

In [3]:
agent = create_agent(
    model='claude-haiku-4-5',
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='claude-haiku-4-5',
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ]
)

In [12]:
response = agent.invoke({
    "messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]
},
{"configurable": {"thread_id": "1"}}
)

In [13]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nUser is exploring fictional/imaginative scenarios about the moon, including questions about a fictional lunar city called "Lunapolis" and its inhabitants.\n\n## SUMMARY\n\nThe conversation began with a legitimate request for pork loin recipes, which was answered with comprehensive culinary information including classic roasts, flavorful preparations, quick weeknight options, and pro cooking tips.\n\nThe conversation then shifted to a series of fictional questions about the moon:\n- User asked about "the capital of the moon" (Lunapolis)\n- Inquired about weather conditions in Lunapolis\n- Asked about cheese miners living in Lunapolis\n- Speculated about a cheese miners\' union strike\n\nThis appears to be either a creative/imaginative exercise or a test of how the assistant responds to fictional premises.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nClarify with the user whether the

## Selective Trimming of Messages

Custom decorator functions can trim specific messages in a conversation. Builtin decorator functions @before_agent and @after_agent allow you to precisely control messages and tool calls that are stored or removed during agent execution.

In [14]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [15]:
agent2 = create_agent(
    model='claude-haiku-4-5',
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [17]:
agent2.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='c9b22d84-b11b-4423-af7d-04928dc167d4'),
  AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='ca8cd92f-38cd-4d41-a7ee-0522403941a8', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='31690732-d305-4e94-b2d4-60a5221101cc'),
  AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='46227f9d-9d10-4195-b054-f7a8abdd12cf', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='85a1cbaa-ef9e-4f37-921b-6a333ea9d756'),
  AIMessage(content="Good question — checking the temperature can help diagnose the issue. Here's what to do:\n\n1. **Feel the device carefully